# Question 11 — Relationship Between Spatial Filtering and Frequency Response

> Note: This question requires knowledge beyond what was covered in class.

Consider a 2D image $f(x,y)$ and a filter $h(x,y)$, where the filtered image is:
$$g(x,y) = f(x,y) * h(x,y)$$

## (a) Convolution Theorem — Spatial vs Frequency Domain

The **Convolution Theorem** states that convolution in the spatial domain corresponds to **pointwise multiplication** in the frequency domain:

$$g(x,y) = f(x,y) * h(x,y) \quad \Longleftrightarrow \quad G(u,v) = F(u,v) \cdot H(u,v)$$

where $F(u,v)$, $H(u,v)$, and $G(u,v)$ are the 2D Discrete Fourier Transforms (DFT) of $f$, $h$, and $g$ respectively.

**Practical meaning:**

Instead of sliding a kernel across every pixel (spatial convolution), we can:
1. Compute $F(u,v) = \mathcal{F}\{f(x,y)\}$ — DFT of the image
2. Compute $H(u,v) = \mathcal{F}\{h(x,y)\}$ — DFT of the filter (its **frequency response**)
3. Multiply: $G(u,v) = F(u,v) \cdot H(u,v)$
4. Recover the result: $g(x,y) = \mathcal{F}^{-1}\{G(u,v)\}$

**Why this matters:**
- Spatial convolution with a large kernel of size $k×k$ costs $O(k^2)$ per pixel
- Frequency domain multiplication costs $O(1)$ per frequency component after the DFT
- For large kernels, the frequency domain approach is significantly faster
- More importantly, it gives us direct insight into **what frequencies a filter passes or blocks**

## (b) Frequency Domain Behavior of Common Filters

### Averaging (Box) Filter

**Spatial behavior:** Replaces each pixel with the average of its neighbourhood — uniform weights.

**Frequency domain:** The box filter's Fourier transform is a **sinc function**:
$$H_{box}(u,v) \propto \text{sinc}(u) \cdot \text{sinc}(v)$$

- Acts as a **low-pass filter** — attenuates high frequencies (fine detail, edges)
- However, the sinc function has **side lobes** that do not go to zero — meaning some high frequencies are only partially suppressed and others are actually amplified at certain frequencies
- This causes **ringing artifacts** (Gibbs phenomenon) around sharp edges
- The abrupt cutoff in spatial domain (uniform rectangular window) creates oscillations in frequency domain

---

### Gaussian Filter

**Spatial behavior:** Weights pixels by a Gaussian bell curve — pixels closer to center have more influence.

**Frequency domain:** The Fourier transform of a Gaussian is also a Gaussian:
$$H_{gauss}(u,v) = \exp\left(-\frac{u^2 + v^2}{2\sigma_f^2}\right), \quad \sigma_f = \frac{1}{2\pi\sigma}$$

- Acts as a **smooth low-pass filter** — gently attenuates high frequencies
- **No side lobes** — the frequency response decays monotonically to zero
- Larger $\sigma$ in spatial domain → smaller $\sigma_f$ in frequency domain → stronger low-pass (more blurring)
- This smooth rolloff is why Gaussian filtering avoids ringing artifacts

---

### Laplacian Filter

**Spatial behavior:** Computes the second derivative of intensity — highlights regions of rapid intensity change (edges).

**Frequency domain:** The Laplacian amplifies frequencies proportional to the squared distance from the origin:
$$H_{lap}(u,v) \propto -(u^2 + v^2)$$

- Acts as a **high-pass filter** — suppresses low frequencies (smooth regions), amplifies high frequencies (edges, fine detail)
- The response grows with frequency — meaning it also amplifies high-frequency **noise**
- Used for edge detection and sharpening, not noise reduction

## (c) Why Gaussian Avoids Ringing Artifacts

**Ringing artifacts** (Gibbs phenomenon) occur when a filter has an abrupt cutoff in the frequency domain. The ideal low-pass filter is a perfect rectangle in frequency space:

$$H_{ideal}(u,v) = \begin{cases} 1 & \sqrt{u^2+v^2} \leq D_0 \\ 0 & \text{otherwise} \end{cases}$$

This sharp cutoff transforms back to a **sinc function** in the spatial domain — which has infinite oscillating tails. These tails cause visible ringing around sharp edges in the filtered image.

**The Gaussian avoids this because:**

| Property | Ideal Low-Pass | Gaussian |
|---|---|---|
| Frequency response shape | Rectangular (sharp cutoff) | Smooth bell curve |
| Spatial domain equivalent | Sinc (oscillating tails) | Gaussian (no tails) |
| Ringing artifacts | Yes — strong | No |
| Edge preservation | Poor | Better than ideal LP |

The Gaussian is its own Fourier transform — a Gaussian in spatial domain is a Gaussian in frequency domain. Since the Gaussian decays **smoothly and monotonically** to zero with no discontinuities, its inverse transform also has no oscillations. This is the fundamental reason it produces artifact-free smoothing.

## (d) Best Filter for High-Frequency Noise Reduction

The most suitable filter for high-frequency noise reduction is the **Gaussian filter**.

### Spatial Domain Argument
- Gaussian weighted averaging gives **more influence to nearby pixels** and less to distant ones
- This is a more physically meaningful smoothing than uniform averaging (box filter)
- It does not amplify any pixel values — purely a weighted average — so it cannot introduce new artifacts
- Unlike the Laplacian, it suppresses rather than amplifies high frequencies

### Frequency Domain Argument
- High-frequency noise occupies the **outer regions** of the frequency spectrum
- The Gaussian frequency response **smoothly attenuates** these high-frequency components
- The degree of attenuation is controlled by $\sigma$ — larger $\sigma$ = stronger suppression
- Unlike the box filter (sinc response), the Gaussian has **no side lobes** so it does not accidentally amplify certain noise frequencies
- Unlike the ideal low-pass filter, it produces **no ringing** at edges

### Why Not the Others?
| Filter | Why not ideal for noise reduction |
|---|---|
| Box filter | Sinc response causes ringing; uniform weights are suboptimal |
| Laplacian | High-pass filter — amplifies noise rather than suppressing it |
| Ideal low-pass | Causes severe ringing artifacts around edges |

**Conclusion:** The Gaussian filter provides the best trade-off between noise suppression (attenuating high frequencies) and spatial quality (no ringing, controlled blurring via $\sigma$). For impulsive noise like salt & pepper specifically, median filtering is preferred — but for Gaussian/random noise, the Gaussian filter is optimal.